# Cross-validation for time series cheat sheet

**What's in here**
- Why shuffled `KFold` lies on autocorrelated data (demonstrated)
- `TimeSeriesSplit(n_splits, test_size, gap)` and what the folds look like
- Expanding vs rolling windows: a walk-forward loop by hand
- `cross_val_score` / `cross_validate`, scoring strings and the sign convention
- `GridSearchCV` / `RandomizedSearchCV` with a pipeline and a time-series CV
- The `gap` (embargo) when the target is h hours ahead
- Learning-curve style table
- Checklist: is my evaluation honest?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).sort_values("time").reset_index(drop=True)
df["hour"]    = df["time"].dt.hour
df["dow"]     = df["time"].dt.dayofweek
df["month"]   = df["time"].dt.month
df["weekend"] = (df["dow"] >= 5).astype(int)
df["hdd"]     = np.clip(15 - df["temp_c"], 0, None)      # heating degrees
df["cdd"]     = np.clip(df["temp_c"] - 22, 0, None)      # cooling degrees
print(df.shape); df.head(3)

(17520, 12)


,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,hour,dow,month,weekend,hdd,cdd
0,2022-01-01 00:00:00+00:00,26858.4,0.11,7.00,0.0,81.83,0,5,1,1,14.89,0.0
1,2022-01-01 01:00:00+00:00,26177.8,-0.18,6.61,0.0,88.21,1,5,1,1,15.18,0.0
2,2022-01-01 02:00:00+00:00,26229.4,-1.11,7.14,0.0,84.71,2,5,1,1,16.11,0.0


## Setup: a day-ahead problem with lagged features

Everything on the right-hand side must be known at time t. The "temperature forecast" here is the *realised* temperature at t+24 - a perfect forecast, which real life does not give you. It is labelled as such; in an interview, say so.

In [2]:
# Day-ahead style setup: at time t, predict consumption at t+24 using only information available at t.
h = 24
d = df.copy()
d["target"] = d["consumption_mwh"].shift(-h)                       # future value
d["lag0"]   = d["consumption_mwh"]                                 # current (known at t)
d["lag24"]  = d["consumption_mwh"].shift(24)
d["lag168"] = d["consumption_mwh"].shift(168)
d["roll24"] = d["consumption_mwh"].rolling(24).mean()              # includes current hour - OK because target is t+24
d["hour_t"] = d["time"].dt.hour                                    # hour of the *target* is the same (h=24)
d["dow_t"]  = (d["time"] + pd.Timedelta(hours=h)).dt.dayofweek
d["weekend_t"] = (d["dow_t"] >= 5).astype(int)
d["temp_fc"] = d["temp_c"].shift(-h)                               # "perfect" temperature forecast for t+24 (flagged!)
hour_d = pd.get_dummies(d["hour_t"], prefix="h", dtype=float)
d = pd.concat([d, hour_d], axis=1).dropna().reset_index(drop=True)
feats = ["lag0", "lag24", "lag168", "roll24", "weekend_t", "temp_fc"] + list(hour_d.columns)
X, y, t = d[feats], d["target"], d["time"]
print(X.shape)

(17328, 30)


## Shuffled KFold vs TimeSeriesSplit

Same model, same features, two CV schemes. A flexible model (random forest) exploits neighbouring hours in the shuffled scheme: the test hour's neighbours (t-1, t+1) are in training and have nearly identical features and target.

In [3]:
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rf = RandomForestRegressor(n_estimators=40, min_samples_leaf=3, n_jobs=-1, random_state=0)
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1))

kf = KFold(n_splits=5, shuffle=True, random_state=0)
ts = TimeSeriesSplit(n_splits=5)
rows = {}
for mname, m in [("random forest", rf), ("ridge", ridge)]:
    rows[mname] = {
        "KFold shuffled RMSE": -cross_val_score(m, X, y, cv=kf, scoring="neg_root_mean_squared_error").mean(),
        "TimeSeriesSplit RMSE": -cross_val_score(m, X, y, cv=ts, scoring="neg_root_mean_squared_error").mean(),
    }
pd.DataFrame(rows).T.round(0)

,KFold shuffled RMSE,TimeSeriesSplit RMSE
random forest,938.0,1053.0
ridge,908.0,1028.0


Both models look about 12% better under shuffled KFold than under a chronological split - and that optimism is pure evaluation artefact: with shuffling, every test hour has its neighbours in training and every fold sees every season, whereas real forecasting always faces a period it has not seen. Here the forest does not beat ridge under either scheme (the relationship is close to linear once hour dummies and lags are in), but the gap between schemes is typically *larger* for flexible models, which memorise neighbours more effectively.

**Interview check:** "Model A beats model B in cross-validation - deploy it?" Which cross-validation? Only a scheme that mimics deployment (train on past, test on future) ranks models honestly. Shuffled scores can rank models differently *and* overstate all of them.

## What TimeSeriesSplit does

Train on everything before the fold, test on the next block. `test_size` fixes the test block length (otherwise folds are equal-sized), `gap` leaves out rows between train and test, `max_train_size` turns the expanding window into a rolling one.

In [4]:
def show_folds(cv, X, t):
    rows = []
    for i, (tr, te) in enumerate(cv.split(X)):
        rows.append({"fold": i, "train_rows": len(tr), "train_end": t.iloc[tr[-1]].date(),
                     "test_start": t.iloc[te[0]].date(), "test_end": t.iloc[te[-1]].date(), "test_rows": len(te)})
    return pd.DataFrame(rows)

print("default (expanding, equal test blocks):")
print(show_folds(TimeSeriesSplit(n_splits=4), X, t).to_string(index=False))
print("\nrolling 1 year train, 2 months test, 24h gap:")
print(show_folds(TimeSeriesSplit(n_splits=4, test_size=24*60, gap=24, max_train_size=24*365), X, t).to_string(index=False))

default (expanding, equal test blocks):
 fold  train_rows  train_end test_start   test_end  test_rows
    0        3468 2022-06-01 2022-06-01 2022-10-23       3465
    1        6933 2022-10-23 2022-10-23 2023-03-17       3465
    2       10398 2023-03-17 2023-03-17 2023-08-08       3465
    3       13863 2023-08-08 2023-08-08 2023-12-30       3465

rolling 1 year train, 2 months test, 24h gap:
 fold  train_rows  train_end test_start   test_end  test_rows
    0        8760 2023-05-03 2023-05-05 2023-07-03       1440
    1        8760 2023-07-02 2023-07-04 2023-09-01       1440
    2        8760 2023-08-31 2023-09-02 2023-10-31       1440
    3        8760 2023-10-30 2023-11-01 2023-12-30       1440


**Pitfall:** the first fold of the default `TimeSeriesSplit` trains on a small slice (here ~3,000 hours = 4 months), which may not even contain a winter. Set `test_size` and check the first fold's `train_end` - or drop it from the average.

## Walk-forward by hand

The most transparent scheme: each month, refit on all data up to the month start (expanding) or on the last N months (rolling), predict the month, move on. Collect the out-of-sample predictions into one series and score *that* - one number over a continuous OOS period, plus a per-month table that shows deterioration.

In [5]:
from sklearn.metrics import root_mean_squared_error, r2_score

def walk_forward(model, X, y, t, start="2022-07-01", window_months=None):
    months = pd.period_range(start, t.iloc[-1], freq="M")
    oos = pd.Series(index=y.index, dtype=float)
    for m in months:
        m_start, m_end = m.start_time.tz_localize("UTC"), m.end_time.tz_localize("UTC")
        tr_mask = t < m_start
        if window_months:
            tr_mask &= t >= (m_start - pd.DateOffset(months=window_months))
        te_mask = (t >= m_start) & (t <= m_end)
        if te_mask.sum() == 0: continue
        model.fit(X[tr_mask], y[tr_mask])
        oos[te_mask] = model.predict(X[te_mask])
    return oos.dropna()

oos_exp  = walk_forward(make_pipeline(StandardScaler(), Ridge(alpha=1)), X, y, t)
oos_roll = walk_forward(make_pipeline(StandardScaler(), Ridge(alpha=1)), X, y, t, window_months=6)
yo = y[oos_exp.index]
print(f"expanding window : RMSE={root_mean_squared_error(yo, oos_exp):.0f}  R2={r2_score(yo, oos_exp):.4f}  ({len(oos_exp)} OOS hours)")
print(f"rolling 6 months : RMSE={root_mean_squared_error(yo, oos_roll):.0f}  R2={r2_score(yo, oos_roll):.4f}")

expanding window : RMSE=928  R2=0.9504  (13152 OOS hours)
rolling 6 months : RMSE=1012  R2=0.9410


In [6]:
per_month = pd.DataFrame({"actual": yo, "expanding": oos_exp, "rolling6": oos_roll, "month": t[oos_exp.index].dt.strftime("%Y-%m").values})
per_month.groupby("month").apply(lambda g: pd.Series({
    "rmse_expanding": root_mean_squared_error(g["actual"], g["expanding"]),
    "rmse_rolling6":  root_mean_squared_error(g["actual"], g["rolling6"]),
    "bias_expanding": (g["expanding"] - g["actual"]).mean(),
}), include_groups=False).round(0).T

month,2022-07,2022-08,2022-09,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12
rmse_expanding,1093.0,1069.0,1065.0,869.0,800.0,832.0,862.0,825.0,793.0,812.0,986.0,1009.0,1057.0,1054.0,868.0,857.0,868.0,867.0
rmse_rolling6,1093.0,1037.0,1017.0,954.0,1355.0,926.0,986.0,829.0,767.0,770.0,885.0,1261.0,1269.0,1022.0,820.0,910.0,1108.0,957.0
bias_expanding,-284.0,-310.0,132.0,320.0,5.0,70.0,-153.0,-27.0,94.0,224.0,472.0,76.0,-412.0,-122.0,336.0,256.0,244.0,221.0


**Interview check:** "Expanding or rolling window?" Expanding uses more data (better for stable relationships), rolling adapts to regime change (better when the relationship drifts). Look at the per-month table: if the rolling window wins consistently in the later months, the data-generating process is drifting.

## cross_val_score and cross_validate

`cross_val_score` returns one score per fold. `cross_validate` takes several metrics and returns a dict (also fit times, and train scores with `return_train_score=True`).

In [7]:
from sklearn.model_selection import cross_validate
ts = TimeSeriesSplit(n_splits=5, test_size=24*60)
cvres = cross_validate(ridge, X, y, cv=ts,
                       scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"],
                       return_train_score=True)
res = pd.DataFrame(cvres)
res.columns = [c.replace("neg_", "") for c in res.columns]
res.round(3)

,fit_time,score_time,test_root_mean_squared_error,train_root_mean_squared_error,test_mean_absolute_error,train_mean_absolute_error,test_r2,train_r2
0,0.009,0.004,-825.698,-898.441,-644.566,-709.136,0.953,0.955
1,0.009,0.005,-990.685,-888.710,-778.988,-698.872,0.936,0.955
2,0.010,0.004,-1073.262,-899.314,-825.128,-706.937,0.929,0.955
3,0.011,0.004,-866.152,-914.390,-706.668,-719.766,0.948,0.954
4,0.015,0.004,-870.975,-909.510,-685.528,-717.517,0.941,0.954


**Pitfall - the sign convention.** sklearn maximises scores, so error metrics come out *negative* (`neg_root_mean_squared_error`). `-scores.mean()` is the RMSE. A "score" of -900 that goes to -850 is an improvement. Forgetting this flips your model ranking.

In [8]:
from sklearn.metrics import get_scorer_names
print([s for s in get_scorer_names() if "error" in s or s.startswith("r2")])

['d2_absolute_error_score', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'r2']


## Hyper-parameter search with a time-series CV

Pipeline + `GridSearchCV`; parameter names use `step__param`. Pass `cv=TimeSeriesSplit(...)`. `refit=True` (default) refits the best model on all the data at the end. `cv_results_` is a dict you should turn into a DataFrame.

In [9]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import loguniform

pipe = make_pipeline(StandardScaler(), Ridge())
grid = GridSearchCV(pipe, param_grid={"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
                    cv=TimeSeriesSplit(n_splits=5, test_size=24*60, gap=24),
                    scoring="neg_root_mean_squared_error", return_train_score=True)
grid.fit(X, y)
print("best:", grid.best_params_, " CV RMSE:", round(-grid.best_score_, 1))
cvr = pd.DataFrame(grid.cv_results_)[["param_ridge__alpha", "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]]
cvr.assign(mean_train_score=-cvr["mean_train_score"], mean_test_score=-cvr["mean_test_score"]).round(1)

best: {'ridge__alpha': 100}  CV RMSE: 922.8


,param_ridge__alpha,mean_train_score,mean_test_score,std_test_score,rank_test_score
0,0.0,902.0,925.7,92.6,5
1,0.1,902.0,925.7,92.6,4
2,1.0,902.0,925.7,92.4,3
3,10.0,902.1,925.0,90.1,2
4,100.0,907.2,922.8,72.2,1
5,1000.0,998.1,968.1,51.8,6


The CV curve is flat: with 17k rows and ~30 well-scaled features, Ridge barely needs regularisation. That is a finding worth stating ("alpha does not matter here") rather than reporting `alpha=10` as if it were tuned.

`RandomizedSearchCV` samples from distributions - better when the grid would be large. `loguniform` for scale parameters.

In [10]:
rs = RandomizedSearchCV(pipe, param_distributions={"ridge__alpha": loguniform(1e-3, 1e4)}, n_iter=8,
                        cv=TimeSeriesSplit(n_splits=3, test_size=24*60), scoring="neg_root_mean_squared_error", random_state=0)
rs.fit(X, y)
pd.DataFrame(rs.cv_results_)[["param_ridge__alpha", "mean_test_score", "rank_test_score"]].sort_values("rank_test_score").head(4).round(2)

,param_ridge__alpha,mean_test_score,rank_test_score
1,101.47,-925.37,1
5,33.21,-931.94,2
2,16.57,-934.26,3
0,6.95,-935.78,4


**Interview check:** "You chose alpha with GridSearchCV and then reported the same CV score as the model's performance. Is that fair?" No - the selection used the test folds, so the best score is optimistically biased. Either hold out a final untouched period, or do nested CV (an outer loop that evaluates the inner search). With one hyper-parameter on a flat curve the bias is small; with dozens of features tried it is not.

## The gap / embargo

Our target at row t is consumption at t+24. The last 24 training rows have targets that fall *inside* the test period, and the first test rows have features (lag0, roll24) that overlap the last training targets. `gap=h` removes that overlap. Without it the leak is mild for 24h; for long horizons or overlapping rolling targets it is not.

In [11]:
def leak_check(cv):
    tr, te = next(cv.split(X))
    last_train_target_time = t.iloc[tr[-1]] + pd.Timedelta(hours=h)
    first_test_time = t.iloc[te[0]]
    return last_train_target_time, first_test_time

for g in [0, 24]:
    a, b = leak_check(TimeSeriesSplit(n_splits=3, gap=g))
    print(f"gap={g:2d}: last train target realised at {a}  |  first test feature time {b}  -> {'OVERLAP' if a > b else 'clean'}")

gap= 0: last train target realised at 2022-07-08 11:00:00+00:00  |  first test feature time 2022-07-07 12:00:00+00:00  -> OVERLAP
gap=24: last train target realised at 2022-07-07 11:00:00+00:00  |  first test feature time 2022-07-07 12:00:00+00:00  -> clean


## Learning-curve style table

How does OOS error change with the amount of training history? Fix the test period (last 3 months), train on the preceding N months.

In [12]:
test_mask = t >= t.iloc[-1] - pd.DateOffset(months=3)
Xte, yte = X[test_mask], y[test_mask]
rows = []
for months in [1, 2, 3, 6, 12, 18]:
    tr_mask = (~test_mask) & (t >= t[test_mask].iloc[0] - pd.DateOffset(months=months))
    m = make_pipeline(StandardScaler(), Ridge(alpha=1)).fit(X[tr_mask], y[tr_mask])
    rows.append({"train_months": months, "train_rows": int(tr_mask.sum()),
                 "train_rmse": root_mean_squared_error(y[tr_mask], m.predict(X[tr_mask])),
                 "test_rmse": root_mean_squared_error(yte, m.predict(Xte))})
pd.DataFrame(rows).round(0)

,train_months,train_rows,train_rmse,test_rmse
0,1,744,715.0,1588.0
1,2,1488,770.0,1875.0
2,3,2208,771.0,1795.0
3,6,4416,880.0,1029.0
4,12,8760,886.0,847.0
5,18,13176,915.0,867.0


If test error keeps falling with more history, get more data. If train error is far below test error, the model overfits (regularise, simplify). If both plateau high, the features are the limit.

## Checklist: is my evaluation honest?

- [ ] Test period is strictly after training period (no shuffle anywhere, including inside `cross_val_score`).
- [ ] Every feature at row t was computable at time t (lags use `shift(k)` with k >= horizon-appropriate values; rolling windows end at or before t; forecasts are the forecast *available* at t, not the realised value).
- [ ] Preprocessing (scaler, imputer, encoder, selection) is inside the pipeline and refit per fold.
- [ ] `gap` >= horizon when targets overlap the next period.
- [ ] Hyper-parameters chosen on folds that are not the ones reported as the final score.
- [ ] The metric is compared with a naive baseline on the *same* test period.
- [ ] Results are reported per period (month/regime), not only as one average.
- [ ] The first fold has enough history to be meaningful.

## Quick reference

| Task | Call |
|---|---|
| Time CV | `TimeSeriesSplit(n_splits=5, test_size=24*30, gap=24, max_train_size=None)` |
| One metric per fold | `cross_val_score(m, X, y, cv=ts, scoring="neg_root_mean_squared_error")` |
| Several metrics | `cross_validate(m, X, y, cv=ts, scoring=[...], return_train_score=True)` |
| Grid | `GridSearchCV(pipe, {"ridge__alpha": [...]}, cv=ts, scoring=...)`; `pd.DataFrame(grid.cv_results_)` |
| Random | `RandomizedSearchCV(pipe, {"ridge__alpha": loguniform(1e-3, 1e3)}, n_iter=20, cv=ts)` |
| Scorer names | `sklearn.metrics.get_scorer_names()` |